In [2]:
!pip install easyocr pdf2image opencv-python-headless
!apt-get install -y poppler-utils

In [3]:
import easyocr
from pdf2image import convert_from_path
import re
import json
import numpy as np
import os
import cv2

In [4]:

# Initialize OCR reader globally to save memory
reader = easyocr.Reader(['en'])

def extract_text_from_pdf(pdf_path):
    """Converts PDF to images and performs OCR on every page."""
    print(f" Processing: {pdf_path}")
    pages = convert_from_path(pdf_path)
    all_tokens = []
    for i, page in enumerate(pages):
        print(f"  Scanning Page {i+1}/{len(pages)}...")
        results = reader.readtext(np.array(page))
        # Store just the text and remove internal spaces
        all_tokens.extend([res[1].strip() for res in results])
    return all_tokens

# Initialize OCR reader globally to save memory
def preprocess_for_ocr(pil_image):
    """
    Uses OpenCV to upscale the image and apply a strict threshold.
    This deletes light watermarks and makes small text bold and readable.
    """
    # Convert PIL Image to numpy array (OpenCV format)
    img_cv = np.array(pil_image)

    # Convert RGB to BGR and then to Grayscale
    img_cv = cv2.cvtColor(img_cv, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)

    # Upscale 2x to make small Option numbers massive for the OCR
    gray = cv2.resize(gray, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)

    # Thresholding: Turns everything darker than 180 to black, and lighter to white.
    # This destroys the background watermarks but keeps the black text.
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)

    return thresh

def process_correct_options(pdf_path):
    """Extracts Correct Answer keys from correct_option.pdf"""
    print(f"\nProcessing Correct Options from: {pdf_path}")
    pages = convert_from_path(pdf_path)
    final_data = {}

    q_regex = r'\b2268\d{8}\b'  # 12 digits
    a_regex = r'\b2268\d{9}\b'  # 13 digits

    for i, page in enumerate(pages):
        prep_img = preprocess_for_ocr(page)
        results = reader.readtext(prep_img)
        tokens = [res[1].replace(" ", "") for res in results]

        temp_q = None
        for clean in tokens:
            if re.fullmatch(q_regex, clean):
                temp_q = clean
            elif re.fullmatch(a_regex, clean) and temp_q:
                final_data[temp_q] = clean
                temp_q = None

    return final_data

def process_response_sheet(pdf_path):
    """
    Extracts Question ID, Options, Status, and Chosen Option page-by-page.
    Prints the result live during the scan.
    """
    print(f"\n Processing Your Responses from: {pdf_path}")
    pages = convert_from_path(pdf_path)
    response_data = {}

    q_regex = re.compile(r'22689578\d{4}')
    opt_regex = re.compile(r'22689530\d{5}')

    for i, page in enumerate(pages):
        print(f"\n Scanning Page {i+1}/{len(pages)}...")

        # Apply OpenCV Magic
        prep_img = preprocess_for_ocr(page)

        # Read text from processed image
        results = reader.readtext(prep_img)
        tokens = [res[1].strip() for res in results]

        j = 0
        while j < len(tokens):
            clean_token = tokens[j].replace(" ", "").replace(":", "")

            # Detect Question ID Block
            if q_regex.search(clean_token):
                q_id = q_regex.search(clean_token).group()

                # Grab surrounding tokens to form a solid text block
                block_tokens = tokens[j:min(j + 35, len(tokens))]
                # Crush into one string for indestructible Regex matching
                block_str = "".join(block_tokens).replace(" ", "").replace(":", "").replace("-", "")

                # Extract Option IDs
                found_options = opt_regex.findall(block_str)
                option_id_list = []
                for opt in found_options:
                    if opt not in option_id_list:
                        option_id_list.append(opt)

                # Detect Status
                status = "Not Answered"
                # If "StatusAnswered" is present and it is NOT "StatusNotAnswered"
                if re.search(r'StatusAnswered', block_str, re.IGNORECASE) and not re.search(r'NotAnswered', block_str, re.IGNORECASE):
                    status = "Answered"

                # Detect Chosen Option Number
                chosen_num = "--"
                chosen_id_val = "--"

                if status == "Answered":
                    # Advanced Regex allowing for OCR typos like "Chosen0ption2" or "ChosenOplion3"
                    chosen_match = re.search(r'[Cc]hosen[A-Za-z0-9]*?([1-4])', block_str)
                    if chosen_match:
                        chosen_num = int(chosen_match.group(1))

                        # Map Number to 13-digit ID
                        try:
                            chosen_id_val = option_id_list[chosen_num - 1]
                        except IndexError:
                            chosen_id_val = "Error: ID not found"

                # Live Print out
                print(f" Found Q_ID: {q_id} | Status: {status} | Chosen: {chosen_num}")

                # Save to dictionary
                response_data[q_id] = {
                    "Question ID": q_id,
                    "Status": status,
                    "All Option IDs": option_id_list,
                    "Chosen Option": chosen_num,
                    "ChosenOptionID": chosen_id_val
                }

                j += 15 # Skip ahead
            else:
                j += 1

    return response_data

def calculate_score(correct_json_data, response_json_data):
    """Compares responses with correct keys and calculates marks."""
    total_score = 0
    correct_count = 0
    wrong_count = 0
    unattempted = 0
    wrong_questions = []
    print("\n" + "="*40)
    print(" FINAL SCORE CALCULATION")
    print("="*40)

    for q_id, resp in response_json_data.items():
        correct_answer_id = correct_json_data.get(str(q_id))

        # Skip if Not Answered
        if resp.get("Status") != "Answered":
            unattempted += 1
            continue

        student_choice_id = resp.get("ChosenOptionID")

        # Compare IDs
        if student_choice_id == correct_answer_id:
            total_score += 5
            correct_count += 1
        else:
            # Check if valid to avoid penalizing OCR failures
            if student_choice_id != "--" and "Error" not in str(student_choice_id):
                total_score -= 1
                wrong_count += 1
                wrong_questions.append({
                    "Question_ID": q_id,
                    "Your_Option": student_choice_id,
                    "Correct_Option": correct_answer_id
                })
            else:
                unattempted += 1

    print(f" Correct (+5): {correct_count}")
    print(f" Wrong (-1):   {wrong_count}")
    print(f" Unattempted:   {unattempted}")
    print("-" * 40)
    print(f" TOTAL SCORE:  {total_score}")
    print("="*40)
    if wrong_questions:
        print("\n INCORRECT QUESTIONS REPORT ")
        print("-" * 40)
        for i, wrong in enumerate(wrong_questions, 1):
            print(f"{i}. Question ID : {wrong['Question_ID']}")
            print(f"   Your Answer : {wrong['Your_Option']}")
            print(f"   Correct Ans : {wrong['Correct_Option']}\n")
    else:
        if correct_count > 0:
            print("\n Excellent! You have NO incorrect answers!")
# --- EXECUTION BLOCK ---

correct_pdf = "correct_answer_pc.pdf"
response_pdf = "response_pc.pdf"
correct_json_file = "correct_answers.json"
response_json_file = "response_pc.json"

# ==========================================
# Step A: Get Correct Answers
# ==========================================
if os.path.exists(correct_json_file):
    print(f" Found existing '{correct_json_file}'. Loading data...")
    with open(correct_json_file, "r") as f:
        correct_map = json.load(f)
elif os.path.exists(correct_pdf):
    print(f"📄 '{correct_json_file}' not found. Processing '{correct_pdf}'...")
    correct_map = process_correct_options(correct_pdf)
    with open(correct_json_file, "w") as f:
        json.dump(correct_map, f, indent=4)
    print(f"✅ Created '{correct_json_file}'.")
else:
    print(f"❌ Error: Missing both '{correct_pdf}' and '{correct_json_file}'!")
    exit()

# ==========================================
# Step B: Get Student Responses
# ==========================================
if os.path.exists(response_json_file):
    print(f"📂 Found existing '{response_json_file}'. Loading data...")
    with open(response_json_file, "r") as f:
        response_map = json.load(f)
elif os.path.exists(response_pdf):
    print(f"📄 '{response_json_file}' not found. Processing '{response_pdf}' with OpenCV OCR...")
    response_map = process_response_sheet(response_pdf)
    with open(response_json_file, "w") as f:
        json.dump(response_map, f, indent=4)
    print(f"✅ Saved detailed student responses to '{response_json_file}'.")
else:
    print(f"❌ Error: Missing both '{response_pdf}' and '{response_json_file}'!")
    exit()

# ==========================================
# Step C: Calculate Marks
# ==========================================
if correct_map and response_map:
    calculate_score(correct_map, response_map)
else:
    print(" Error: Missing data to calculate score.")